In [1]:
%load_ext autoreload
%autoreload 2

import sys
import pandas as pd
sys.path.insert(1, '../')

In [2]:
import pyFBS
from pyFBS.utility import *
import pickle
import pyvista as pv

## ODS - sensor animation


In [3]:
view3D = pyFBS.display.view3D()

Add a structure from .stl file to the 3D 

In [4]:
receiver = "../data/AM_automotive_testbench/STL/receiver.stl"
view3D.add_stl(receiver,name = "receiver",color = "#D3D3D3",opacity = 0.2)

In [5]:
roll_mount = "../data/AM_automotive_testbench/STL/roll_mount.stl"
view3D.add_stl(roll_mount,name = "roll_mount",color = "#8FB1CC",opacity = 0.2)

transmission_mount = "../data/AM_automotive_testbench/STL/transmission_mount.stl"
view3D.add_stl(transmission_mount,name = "transmission_mount",color = "#8FB1CC",opacity = 0.2)

engine_mount = "../data/AM_automotive_testbench/STL/engine_mount.stl"
view3D.add_stl(engine_mount,name = "engine_mount",color = "#8FB1CC",opacity = 0.2)

In [6]:
ts = "../data/AM_automotive_testbench/STL/ts.stl"
view3D.add_stl(ts,name = "ts",color = "#D3D3D3",opacity = 0.2)

#### Accelerometers
Add accelerometers from .xlsx file together with appropriate labels

In [7]:
AB = "../data/AM_automotive_testbench/Measurements/receiver_substitute/sub_receiver_substitute.xlsx"
df = pd.read_excel(AB, sheet_name='Sensors')

view3D.show_acc(df)
#view3D.label_acc(df)
df

,Name,Description,Type,Size,NodeNumber,Grouping,Quantity,Unit,Position_1,Position_2,Position_3,Orientation_1,Orientation_2,Orientation_3
0,Sensor 1,On substructure receiver substitute,Kistler 50g,0.0102,1,1,Acceleration,m/s^2,0.090778,0.421992,0.286750,0,0,-90
1,Sensor 2,On substructure receiver substitute,Kistler 50g,0.0102,2,1,Acceleration,m/s^2,0.112689,0.442753,0.258348,90,0,-90
2,Sensor 3,On substructure receiver substitute,Kistler 50g,0.0102,3,1,Acceleration,m/s^2,0.093525,0.375674,0.240036,0,-90,90
3,Sensor 4,On substructure receiver substitute,Kistler 50g,0.0102,4,2,Acceleration,m/s^2,0.194314,0.298105,0.216550,180,0,-90
4,Sensor 5,On substructure receiver substitute,Kistler 50g,0.0102,5,2,Acceleration,m/s^2,0.143966,0.259899,0.197206,90,0,0
5,Sensor 6,On substructure receiver substitute,Kistler 50g,0.0102,6,2,Acceleration,m/s^2,0.227000,0.271946,0.177905,-90,0,-90
6,Sensor 7,On substructure receiver substitute,Kistler 50g,0.0102,7,3,Acceleration,m/s^2,0.347908,0.376833,0.274001,0,90,90
7,Sensor 8,On substructure receiver substitute,Kistler 50g,0.0102,8,3,Acceleration,m/s^2,0.332241,0.362723,0.226550,180,0,-90
8,Sensor 9,On substructure receiver substitute,Kistler 50g,0.0102,9,3,Acceleration,m/s^2,0.344095,0.286222,0.276692,0,90,0
9,Sensor 10,On substructure receiver substitute,Kistler 50g,0.0102,10,20,Acceleration,m/s^2,0.141951,0.481895,0.243394,0,-90,-90


Generate channels from sensors

In [8]:
columns_chann = ["Name","Description","Type","DirectionLabel","Quantity","Unit","Component","NodeNumber","Grouping","Position_1","Position_2","Position_3","Direction_1","Direction_2","Direction_3"]
df_ch = pd.DataFrame(columns = columns_chann)


df = pd.read_excel(AB, sheet_name='Sensors')
axes = ["x","y","z"]
for s,angle in enumerate(df[["Orientation_1","Orientation_2","Orientation_3"]].to_numpy()):
    R = eulerAnglesToRotationMatrix(angle*np.pi/180)
    for i in range(3):
        data_chn = np.asarray([[df["Name"][s] + axes[i],None, None, None,None, None,None, None, df["Grouping"][s],df["Position_1"][s],df["Position_2"][s],df["Position_3"][s],R[i][0],R[i][1],R[i][2]]])
        df_row = pd.DataFrame(data = data_chn,columns = columns_chann)
        df_ch = df_ch.append(df_row)

#### Channels
Add corresponding channels from .xlsx file together with appropriate labels

In [9]:
df = pd.read_excel(AB, sheet_name='ChannelsMod')

view3D.show_chn(df_ch)
view3D.label_chn(df_ch)


#### Impacts
Add impacts from .xlsx with appropriate labels

In [10]:
df = pd.read_excel(AB, sheet_name='Impacts')

view3D.show_imp(df)
view3D.label_imp(df)

In [11]:
file_name = "../data/AM_automotive_testbench/Measurements/receiver_substitute/run1_data.npy"

Y = np.load(file_name)
file_name = "../data/AM_automotive_testbench/Measurements/receiver_substitute/run1_freq.npy"

freq = np.load(file_name)

Y_tran = Y[:-1,:,:]

In [12]:
select_in = 0
freq_sel = 150


df = pd.read_excel(AB, sheet_name='Sensors')

n_sen = len(df)
n_ax = 3

empty = np.zeros((n_sen,n_ax),dtype = complex)

for i in range(n_sen):
    for j in range(n_ax):
        empty[i,j] = Y_tran[(i)*3+j,select_in,freq_sel]

df = pd.read_excel(AB, sheet_name='Sensors')
new = np.zeros_like(empty,dtype = complex)

for s,angle in enumerate(df[["Orientation_1","Orientation_2","Orientation_3"]].to_numpy()):
    R = eulerAnglesToRotationMatrix(angle*np.pi/180)
    new[s,:] = empty[s,:]@R
    


In [13]:
view3D.take_gif = False
view3D.gif_dir = "file.gif"

In [14]:
mode_dict = dict_animation(new,"object",object_list = view3D.global_acc)
mode_dict["freq"] = freq[freq_sel]
view3D.add_objects_animation(mode_dict,run_animation = True,add_note= True)